# DocMind - RAG Pipeline Notebook
Step-by-step walkthrough of the full pipeline.

In [1]:
import sys
sys.path.insert(0, '..')

from langchain_community.document_loaders import PDFPlumberLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

c:\docmind_langchain\docmind_langchain\venv_assesment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Document

In [2]:
DOC_PATH ='../data\AI Training Document.pdf'

loader = PDFPlumberLoader(DOC_PATH)
docs = loader.load()

print(f'Pages loaded: {len(docs)}')
print(f'First 300 chars:\n{docs[0].page_content[:300]}')

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\91987\AppData\Local\Temp\ipykernel_6512\2482514182.py:1: SyntaxWarning: invalid escape sequence '\A'
  DOC_PATH ='../data\AI Training Document.pdf'


Pages loaded: 20
First 300 chars:
User Agreement
1. Introduction
This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms
posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms
on which eBay offers you access to and use of our Services. You can


## Step 2: Split into Chunks

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=['\n\n', '\n', '.', '!', '?', ' ', '']
)

chunks = splitter.split_documents(docs)
print(f'Total chunks: {len(chunks)}')
print(f'Chunk size range: {min(len(c.page_content) for c in chunks)} - {max(len(c.page_content) for c in chunks)} chars')

# Show sample
for i in [0, 5, -1]:
    print(f'\n[Chunk {i}]')
    print(chunks[i].page_content[:200])

Total chunks: 105
Chunk size range: 108 - 800 chars

[Chunk 0]
User Agreement
1. Introduction
This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms
posted on and in our sites, applications, tools, and services (collective

[Chunk 5]
or legality of items advertised; the truth or accuracy of users' content or listings; the ability of sellers

[Chunk -1]
complaints shall be decided by an independent arbitrator in accordance with this User Agreement.
Buyers and sellers further agree to submit to the jurisdiction of the State of Illinois for complaints



## Step 3 + 4: Embed and Store in FAISS

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local('../vectordb')
print('FAISS index built and saved!')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3432.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index built and saved!


## Step 5: Retrieval Test

In [5]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

test_queries = [
    'What data is collected from users?',
    'What are the user obligations?',
    'What is the refund policy?',
]

for q in test_queries:
    results = retriever.invoke(q)
    print(f'\nQ: {q}')
    for i, r in enumerate(results):
        print(f'  Chunk {i}: {r.page_content[:120]}...')


Q: What data is collected from users?
  Chunk 0: messaging platforms (including but not limited to chat and email channels), including messages
between users, to: (i) de...
  Chunk 1: • take any action that may undermine the feedback or ratings systems (our Feedback policies);
• transfer your eBay accou...
  Chunk 2: • number of listings matching the buyer's query.
• To drive a positive user experience, a listing may not appear in some...

Q: What are the user obligations?
  Chunk 0: • take any action that may undermine the feedback or ratings systems (our Feedback policies);
• transfer your eBay accou...
  Chunk 1: to act on behalf of such business and bind the business to this User Agreement. Such account is
owned and controlled by ...
  Chunk 2: to sell items; the ability of buyers to pay for items; or that a buyer or seller will actually complete a
transaction or...

Q: What is the refund policy?
  Chunk 0: perpetrated by you, or charges necessary to recoup amounts refunded to you

## Step 6: Full RAG Answer with Groq

In [ ]:
GROQ_API_KEY = 'PUT YOUR API KEY HERE <3'

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name='llama-3.3-70b-versatile',
    temperature=0.2,
    max_tokens=512
)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer using ONLY the context provided. If not found, say so.'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])

def ask(question):
    docs = retriever.invoke(question)
    context = '\n\n'.join(d.page_content for d in docs)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({'context': context, 'question': question})
    return answer, docs

# Test queries
eval_questions = [
    'What are the main terms and conditions?',
    'How is user data protected?',
    'Can users cancel their subscription?',
    'What is the governing law?',
    'What happens if a user violates the terms?',
]

for q in eval_questions:
    ans, srcs = ask(q)
    print(f'\n{"="*50}')
    print(f'Q: {q}')
    print(f'A: {ans[:300]}')
    print(f'Sources: {len(srcs)} chunks used')


Q: What are the main terms and conditions?
A: The context provided does not explicitly state the main terms and conditions. However, it mentions that credit and debit card issuers, credit and debit card networks, and payments services providers may have their own terms and conditions. Additionally, it discusses the terms of an Agreement to Arbi
Sources: 3 chunks used

Q: How is user data protected?
A: The context does not provide a direct answer on how user data is protected. However, it does mention that eBay may scan and analyze messages sent through their messaging tools to detect and prevent fraudulent activity, and that they may store message contents. Additionally, it outlines certain rules
Sources: 3 chunks used

Q: Can users cancel their subscription?
A: The context does not mention users canceling their subscription. It discusses order cancellations, which are subject to eBay's Order cancellation policy, but does not provide information about canceling a subscription.
Sources